# Quick start

In lumped-element experiments, a static magnetic field lets axion dark matter drive oscillating flux through a pickup circuit.
In this tutorial we build a toroidal axion detector, compute its signal coupling and pickup inductance,
and inspect the fields. The [installation guide](../docs/guides/install.md) covers setup; the cells below should be run in order.

## Geometry and field solve

`templates` lists the available designs. `template` creates a toroid with dimensions
in metres; `set` changes a parameter. `solve` computes the static magnet and pickup
fields and writes results under `runs/nb_toroid`.

The summary reports coupling, inductance, stored energy, and field peaks.

We import axionFEM and display its version and available designs to check the setup.

In [1]:
import axionfem as af

print("axionFEM", af.__version__)
[t["name"] for t in af.templates()]

axionFEM 0.1.0


['toroid', 'solenoid', 'core']

We create a toroid and set its `y` parameter to define the detector geometry.

In [2]:
d = af.template("toroid", R=2.0, H=2.0)
d.set(y=0.6)
d

Design('toroid'; H=2.0, NI=1000000.0, R=2.0, f=0.02*R, g=0.005*R, ts=0.005*R, tw=0.01*R, y=0.6)

We solve for the static magnet and pickup fields and print the summary to obtain the signal coupling and pickup inductance.

In [3]:
res = d.solve("runs/nb_toroid", verbose=0)
print(res.summary().split("\n  benchmark ")[0])    # the figures of merit (per-benchmark lines follow in the full summary)

design 'toroid'
  Psi      = 3.088e-07 H T m^2  (0.2458 T m^3 natural)
  L_p      = 1.458e-06 H
  W_mag    = 1.046e+05 J
  F        = 4.276e-15 T^4 H^2 m^8
  g        = 0.09511
  |B|max   = 0.1667 T  at r = 1.200 m, z = 0.407 m (winding)
  SC surface peak [sheath] = 2.407e-18 T


## Field inspection

`show` opens the viewer. `quantities` lists fields; `quantity` selects one and
`scale` sets its color scale. `pickup_Bmag` is the magnetic field per ampere of
pickup current. The last cell shows a meridional section through the toroid.

We open the viewer to inspect the computed fields.

In [4]:
w = res.show(height=520)
w

We list the available field quantities to choose a view.

In [5]:
w.quantities

['Bmag', 'Br', 'Bz', 'Bphi', 'psi', 'pickup_Bmag', 'pickup_psi']

We display `pickup_Bmag` on a logarithmic scale to inspect the pickup field.

In [6]:
# The field of a unit readout current, on a log scale.
w.quantity = "pickup_Bmag"
w.scale = "log"
w.quantity

'pickup_Bmag'

We display `Bphi` in a meridional section to show its spatial distribution.

In [7]:
# The meridional section, as a second widget.
res.show(mode="section", quantity="Bphi", height=420)

## Results and exports

`Psi` is the signal coupling in H T m², `L_p` the pickup inductance in H, and `F`
the scan figure of merit. $\mathcal F/\mathcal F_{\rm req}\geq1$ meets the selected scan target (by default an illustrative one: DFSZ over
120.9 kHz–1 MHz in three years);
the `optimistic` preset, an illustrative benchmark, assumes noise below the standard quantum limit. The
benchmark also reports the stored energy and peak field the design needs to reach the requirement.

`sample` evaluates fields at cylindrical coordinates in metres. HTML and PNG
exports save views. Mesh convergence should be checked before using numerical values for
design decisions.

We report `Psi`, `L_p`, `F`, and the `optimistic` benchmark ratio, energy and peak field at the requirement.

In [8]:
print(f"Psi = {res.Psi:.4g} H T m^2   L_p = {res.L_p:.4g} H   F = {res.F:.4g}")
b = res.benchmark("optimistic")
print(f"F/F_req = {b['F_over_F_req']:.4g}   at the requirement: W = {b['W_mag_at_req_J']:.4g} J, "
      f"B_max = {b['B_max_at_req_T']:.4g} T")

Psi = 3.088e-07 H T m^2   L_p = 1.458e-06 H   F = 4.276e-15
F/F_req = 2.594e-07   at the requirement: W = 2.054e+08 J, B_max = 7.385 T


We sample the field at three midplane points to compare values at different radii.

In [9]:
res.sample([0.5, 1.5, 2.5], [0.0, 0.0, 0.0])  # B at three points on the midplane

{'Br': array([0., 0., 0.]),
 'Bz': array([0., 0., 0.]),
 'Bphi': array([0.        , 0.13333333, 0.        ]),
 'psi': array([0., 0., 0.]),
 'pickup_Br': array([ 2.10286921e-12,  0.00000000e+00, -1.83508211e-12]),
 'pickup_Bz': array([ 3.40266518e-07,  0.00000000e+00, -2.29303163e-08]),
 'pickup_Bphi': array([0., 0., 0.]),
 'pickup_psi': array([4.22068356e-08, 2.32124557e-07, 2.06152264e-07]),
 'region_attr': array([1., 1., 1.])}

We export an HTML page and a PNG section view to share the field results.

In [10]:
# A standalone page to share, and a PNG figure.
print(res.export_html("runs/nb_toroid/toroid.html"))
print(res.png("runs/nb_toroid/toroid.png", view="section", quantity="Bphi"))

runs/nb_toroid/toroid.html
runs/nb_toroid/toroid.png


## Full 3D

The commented example uses `to_3d` to revolve the geometry. Running it requires
an MPI-enabled build, as described in [installation](../docs/guides/install.md).

We can uncomment the example below to solve the revolved geometry in 3D and view `Bmag`.

In [11]:
# from axionfem.convert import to_3d
#
# d3 = to_3d(d, mesh={"size": 0.4, "surface_size": 0.13, "order": 1})
# res3 = d3.solve("runs/nb_toroid_3d", verbose=1)
# res3.show(quantity="Bmag")

The [lumped-element walkthrough](lumped_element_demo.ipynb)
covers the physics and scan calculation, and [Visualization](../docs/guides/visualization.md)
covers viewing and exporting fields.